In [2]:
import glob
import os
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Import the 3 RV CSVs as separate dataframes
# ---------------------------------------------------------------------------
# Map each CSV filename to a clean, asset-prefixed column name so the shared
# "Realized Variance (5-min)" column doesn't collide on merge.
CSV_ASSET_MAP = {
    "GL_gold_5_min_RV_data.csv": "RV_gold",   # <-- prediction target
    "CL_Crude_5_min_RV_data.csv": "RV_crude",
    "ES_mini_5_min_RV_data.csv": "RV_ES",
}

csv_frames = {}
for path in sorted(glob.glob("*_RV_data.csv")):
    fname = os.path.basename(path)
    col = CSV_ASSET_MAP[fname]
    df = pd.read_csv(path)
    # Canonical date key: tz-naive datetime64[ns] at midnight (strips any time part)
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    df = df.set_index("Date")
    df = df.rename(columns={"Realized Variance (5-min)": col})[[col]]
    csv_frames[col] = df
    print(f"{fname:<32} -> {col:<9} rows={len(df):>5}  {df.index.min().date()} .. {df.index.max().date()}")

gold_rv, crude_rv, es_rv = csv_frames["RV_gold"], csv_frames["RV_crude"], csv_frames["RV_ES"]

# ---------------------------------------------------------------------------
# 2. Load the 4th dataframe: GVZ (gold vol index) from parquet
# ---------------------------------------------------------------------------
gvz = pd.read_parquet("GVZ_daily.parquet").rename(columns={"close": "GVZ_close"})
# Normalize the index the SAME way as the CSVs so all four keys are identical
gvz.index = pd.to_datetime(gvz.index).normalize()
gvz.index.name = "Date"
gvz = gvz[["GVZ_close"]]
print(f"{'GVZ_daily.parquet':<32} -> {'GVZ_close':<9} rows={len(gvz):>5}  {gvz.index.min().date()} .. {gvz.index.max().date()}")


CL_Crude_5_min_RV_data.csv       -> RV_crude  rows= 4239  2010-01-04 .. 2026-05-29
ES_mini_5_min_RV_data.csv        -> RV_ES     rows= 4243  2010-01-04 .. 2026-05-29
GL_gold_5_min_RV_data.csv        -> RV_gold   rows= 4231  2010-01-04 .. 2026-05-29
GVZ_daily.parquet                -> GVZ_close rows= 4202  2009-09-18 .. 2026-06-05


In [3]:
# ---------------------------------------------------------------------------
# 3. Verify all four date keys are comparable (same dtype, tz-naive)
# ---------------------------------------------------------------------------
all_frames = {"RV_gold": gold_rv, "RV_crude": crude_rv, "RV_ES": es_rv, "GVZ_close": gvz}
for name, df in all_frames.items():
    assert df.index.dtype == "datetime64[ns]", f"{name}: index dtype is {df.index.dtype}, expected datetime64[ns]"
    assert df.index.tz is None, f"{name}: index is tz-aware, expected tz-naive"
print("OK: all 4 indexes are tz-naive datetime64[ns] -> join keys are comparable")

# ---------------------------------------------------------------------------
# 4. Merge all 4 on the normalized date index with an INNER join
#    -> keeps only dates present in ALL four datasets
# ---------------------------------------------------------------------------
merged = pd.concat(
    [gold_rv, crude_rv, es_rv, gvz], axis=1, join="inner"
).sort_index()

# ---------------------------------------------------------------------------
# 5. Sanity checks / report
# ---------------------------------------------------------------------------
print(f"\nMerged shape: {merged.shape}")
print(f"Date range:   {merged.index.min().date()} .. {merged.index.max().date()}")
print(f"NaNs:         {int(merged.isna().sum().sum())}  (inner join -> expect 0)")
print(f"Monotonic:    {merged.index.is_monotonic_increasing}")
print("\nDates dropped per source (source rows -> kept in merge):")
for name, df in all_frames.items():
    print(f"  {name:<10} {len(df):>5} -> {len(merged):>5}  (dropped {len(df) - len(merged)})")

merged.head()


OK: all 4 indexes are tz-naive datetime64[ns] -> join keys are comparable

Merged shape: (4114, 4)
Date range:   2010-01-04 .. 2026-05-29
NaNs:         0  (inner join -> expect 0)
Monotonic:    True

Dates dropped per source (source rows -> kept in merge):
  RV_gold     4231 ->  4114  (dropped 117)
  RV_crude    4239 ->  4114  (dropped 125)
  RV_ES       4243 ->  4114  (dropped 129)
  GVZ_close   4202 ->  4114  (dropped 88)


,RV_gold,RV_crude,RV_ES,GVZ_close
Date,,,,
2010-01-04,17.105615,23.312992,10.116760,24.59
2010-01-05,17.374091,22.024859,9.973864,23.34
2010-01-06,17.547456,33.529872,9.444522,23.90
2010-01-07,15.362665,21.070363,10.409856,23.32
2010-01-08,20.881988,24.087747,11.200218,22.04


In [4]:
merged.tail(20)

,RV_gold,RV_crude,RV_ES,GVZ_close
Date,,,,
2026-05-01,18.294799,58.235519,8.660999,26.43
2026-05-04,23.612901,80.709108,13.154332,27.92
2026-05-05,18.906543,64.585180,8.086391,26.61
2026-05-06,21.275878,103.017851,10.421337,26.34
2026-05-07,25.992364,90.776481,12.468201,27.03
2026-05-08,15.881449,46.944555,6.464836,26.48
2026-05-11,19.807367,49.741993,8.174391,27.48
2026-05-12,20.213826,36.780080,10.431980,27.16
2026-05-13,17.577705,34.081518,9.168966,26.56
